In [34]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

In [2]:
LOG_DIR = "../logs/"

AGG_LOG = "he-aggregation-service"
CLIENT1_LOG = "he-client-1-go"
CLIENT2_LOG = "he-client-2-go"

In [3]:
agg_time = pd.read_csv(f"{LOG_DIR}{AGG_LOG}-time.csv")
client1_time = pd.read_csv(f"{LOG_DIR}{CLIENT1_LOG}-time.csv")
client2_time = pd.read_csv(f"{LOG_DIR}{CLIENT2_LOG}-time.csv")

agg_time["Service"] = "AggregationService"
client1_time["Service"] = "Client1"
client2_time["Service"] = "Client2"

client1_time.head()

,Date,EndTime,Name,Runtime,Service
0,2025/07/22,14:41:25.975143,DataSpaceClientService - RegisterClient,13.205041ms,Client1
1,2025/07/22,14:41:25.983500,HEService - SetParameters,8.294416ms,Client1
2,2025/07/22,14:41:30.996970,HEService - PartialShareAggregation,8.583µs,Client1
3,2025/07/22,14:41:31.011173,EncryptionHandler - handleReceivePublicKey,1.149375ms,Client1
4,2025/07/22,14:41:31.037810,HEService - PartialRelinKeyAggregation,2.379958ms,Client1


In [20]:
data_joined = pd.concat(
    [agg_time, client1_time, client2_time],
    axis=0)

data_joined = data_joined.reset_index(drop=True)

data_joined

,Date,EndTime,Name,Runtime,Service
0,2025/07/22,14:41:25.964356,addClient,1.833µs,AggregationService
1,2025/07/22,14:41:31.197952,startEncryptionSetupPhaseFor,5.233553417s,AggregationService
2,2025/07/22,14:41:34.049860,addClient,833ns,AggregationService
3,2025/07/22,14:41:39.414548,startEncryptionSetupPhaseFor,5.36460525s,AggregationService
4,2025/07/22,14:41:53.083988,requestClientTraining,4.722084ms,AggregationService
5,2025/07/22,14:42:24.106563,aggregateWeights,49.423458ms,AggregationService
6,2025/07/22,14:42:24.485907,initiateKeySwitchGeneration,379.262666ms,AggregationService
7,2025/07/22,14:42:27.629453,updateClientModels,2.239003917s,AggregationService
8,2025/07/22,14:42:32.191933,requestClientTraining,3.158125ms,AggregationService
9,2025/07/22,14:42:51.979431,requestClientTraining,1.941917ms,AggregationService


In [21]:
def parse_duration_to_ms(time_str):
    """Parse milliseconds, microseconds, nanoseconds or seconds from a string."""
    if "ms" in time_str:
        return float(time_str.replace("ms", "").strip())
    elif "us" in time_str:
        return float(time_str.replace("us", "").strip()) / 1000.0
    elif "µs" in time_str:
        return float(time_str.replace("µs", "").strip()) / 1000.0
    elif "ns" in time_str:
        return float(time_str.replace("ns", "").strip()) / 1_000_000.0
    elif "s" in time_str:
        return float(time_str.replace("s", "").strip()) * 1000.0
    else:
        raise ValueError(f"Unknown time format: {time_str}")


def create_timeline_offset(df):
    """Create a timeline with an offset for the x-axis."""
    min_time = df["EndTime"].min()
    df["StartMs"] = (pd.to_datetime(df["EndTime"], format="%H:%M:%S.%f") - pd.to_datetime(min_time, format="%H:%M:%S.%f")).dt.total_seconds() * 1000
    
    return df
    
#agg_time["runtime_ms"] = agg_time["Runtime"].apply(parse_duration_to_ms)
#client1_time["runtime_ms"] = client1_time["Runtime"].apply(parse_duration_to_ms)
#client2_time["runtime_ms"] = client2_time["Runtime"].apply(parse_duration_to_ms)
data_joined["runtime_ms"] = data_joined["Runtime"].apply(parse_duration_to_ms)

#agg_time = create_timeline_offset(agg_time)
#client1_time = create_timeline_offset(client1_time)
#client2_time = create_timeline_offset(client2_time)
data_joined = create_timeline_offset(data_joined)

data_joined["EndMs"] = data_joined["StartMs"] + data_joined["runtime_ms"]

data_joined = data_joined.sort_values(by=["StartMs", "EndMs"])
data_joined.reset_index(drop=True, inplace=True)
data_joined

,Date,EndTime,Name,Runtime,Service,runtime_ms,StartMs,EndMs
0,2025/07/22,14:41:25.964356,addClient,1.833µs,AggregationService,0.001833,0.000,0.001833
1,2025/07/22,14:41:25.975143,DataSpaceClientService - RegisterClient,13.205041ms,Client1,13.205041,10.787,23.992041
2,2025/07/22,14:41:25.983500,HEService - SetParameters,8.294416ms,Client1,8.294416,19.144,27.438416
3,2025/07/22,14:41:30.996970,HEService - PartialShareAggregation,8.583µs,Client1,0.008583,5032.614,5032.622583
4,2025/07/22,14:41:31.011173,EncryptionHandler - handleReceivePublicKey,1.149375ms,Client1,1.149375,5046.817,5047.966375
5,2025/07/22,14:41:31.037810,HEService - PartialRelinKeyAggregation,2.379958ms,Client1,2.379958,5073.454,5075.833958
6,2025/07/22,14:41:31.197952,startEncryptionSetupPhaseFor,5.233553417s,AggregationService,5233.553417,5233.596,10467.149417
7,2025/07/22,14:41:34.049860,addClient,833ns,AggregationService,0.000833,8085.504,8085.504833
8,2025/07/22,14:41:34.060062,DataSpaceClientService - RegisterClient,12.486834ms,Client2,12.486834,8095.706,8108.192834
9,2025/07/22,14:41:34.068452,HEService - SetParameters,8.336709ms,Client2,8.336709,8104.096,8112.432709


In [81]:
print(pio.templates)

pio.templates["research"] = go.layout.Template(
    layout=dict(
        font=dict(color="#000000"),
        paper_bgcolor="#ffffff",
        plot_bgcolor="#ffffff",
        hovermode="closest",
        xaxis=dict(
            tickangle=-45,
            showline=True,
            linewidth=1,
            linecolor="#373737",
            ticks="inside",
            showgrid=True,
            gridcolor="#373737",
            zeroline=True,
            zerolinecolor="#373737",
            zerolinewidth=1,
            mirror=True
        ),
        yaxis=dict(
            showline=True,
            linewidth=1,
            linecolor="#373737",
            ticks="inside",
            showgrid=True,
            gridcolor="#373737",
            zeroline=True,
            zerolinecolor="#373737",
            zerolinewidth=1,
            mirror=True
        )
    )
)

pio.templates.default = "plotly_white+research"

Templates configuration
-----------------------
    Default template: 'plotly_white+research'
    Available templates:
        ['ggplot2', 'seaborn', 'simple_white', 'plotly',
         'plotly_white', 'plotly_dark', 'presentation', 'xgridoff',
         'ygridoff', 'gridon', 'none', 'research']



In [82]:
import plotly.express as px
import plotly.graph_objects as go


df = px.data.tips()
fig = px.bar(data_joined[data_joined["runtime_ms"] >= 100], base="StartMs", x="runtime_ms", y="Service", color='Name', orientation='h',
             hover_data=["EndTime", "StartMs", "runtime_ms"], color_discrete_sequence=px.colors.qualitative.Safe,
             height=400)
fig.show()

In [83]:
# remove all waiting times

data_joined.at[data_joined.index[0], "Start_without_offset"] = data_joined.at[data_joined.index[0], "StartMs"]
data_joined.at[data_joined.index[0], "End_without_offset"] = data_joined.at[data_joined.index[0], "EndMs"]

for i in range(1, len(data_joined)):
    data_joined.at[data_joined.index[i], "Start_without_offset"] = data_joined.iloc[i-1]["End_without_offset"]
    data_joined.at[data_joined.index[i], "End_without_offset"] = data_joined.at[data_joined.index[i], "Start_without_offset"] + data_joined.at[data_joined.index[i], "runtime_ms"]
data_joined

,Date,EndTime,Name,Runtime,Service,runtime_ms,StartMs,EndMs,Start_without_offset,End_without_offset
0,2025/07/22,14:41:25.964356,addClient,1.833µs,AggregationService,0.001833,0.000,0.001833,0.000000,0.001833
1,2025/07/22,14:41:25.975143,DataSpaceClientService - RegisterClient,13.205041ms,Client1,13.205041,10.787,23.992041,0.001833,13.206874
2,2025/07/22,14:41:25.983500,HEService - SetParameters,8.294416ms,Client1,8.294416,19.144,27.438416,13.206874,21.501290
3,2025/07/22,14:41:30.996970,HEService - PartialShareAggregation,8.583µs,Client1,0.008583,5032.614,5032.622583,21.501290,21.509873
4,2025/07/22,14:41:31.011173,EncryptionHandler - handleReceivePublicKey,1.149375ms,Client1,1.149375,5046.817,5047.966375,21.509873,22.659248
5,2025/07/22,14:41:31.037810,HEService - PartialRelinKeyAggregation,2.379958ms,Client1,2.379958,5073.454,5075.833958,22.659248,25.039206
6,2025/07/22,14:41:31.197952,startEncryptionSetupPhaseFor,5.233553417s,AggregationService,5233.553417,5233.596,10467.149417,25.039206,5258.592623
7,2025/07/22,14:41:34.049860,addClient,833ns,AggregationService,0.000833,8085.504,8085.504833,5258.592623,5258.593456
8,2025/07/22,14:41:34.060062,DataSpaceClientService - RegisterClient,12.486834ms,Client2,12.486834,8095.706,8108.192834,5258.593456,5271.080290
9,2025/07/22,14:41:34.068452,HEService - SetParameters,8.336709ms,Client2,8.336709,8104.096,8112.432709,5271.080290,5279.416999


In [84]:
df = px.data.tips()
fig = px.bar(data_joined, base="Start_without_offset", x="runtime_ms", y="Service", color='Name', orientation='h',
             hover_data=["EndTime", "StartMs", "runtime_ms"], color_discrete_sequence=px.colors.qualitative.Safe,
             height=400)
fig.show()

In [85]:
data_joined.reset_index(drop=True, inplace=True)

first_occ_encrypt = data_joined.loc[data_joined["Name"] == "requestClientTraining"].index[0]
print(first_occ_encrypt)

initializiation_data = data_joined[:first_occ_encrypt]
initializiation_data

17


,Date,EndTime,Name,Runtime,Service,runtime_ms,StartMs,EndMs,Start_without_offset,End_without_offset
0,2025/07/22,14:41:25.964356,addClient,1.833µs,AggregationService,0.001833,0.000,0.001833,0.000000,0.001833
1,2025/07/22,14:41:25.975143,DataSpaceClientService - RegisterClient,13.205041ms,Client1,13.205041,10.787,23.992041,0.001833,13.206874
2,2025/07/22,14:41:25.983500,HEService - SetParameters,8.294416ms,Client1,8.294416,19.144,27.438416,13.206874,21.501290
3,2025/07/22,14:41:30.996970,HEService - PartialShareAggregation,8.583µs,Client1,0.008583,5032.614,5032.622583,21.501290,21.509873
4,2025/07/22,14:41:31.011173,EncryptionHandler - handleReceivePublicKey,1.149375ms,Client1,1.149375,5046.817,5047.966375,21.509873,22.659248
5,2025/07/22,14:41:31.037810,HEService - PartialRelinKeyAggregation,2.379958ms,Client1,2.379958,5073.454,5075.833958,22.659248,25.039206
6,2025/07/22,14:41:31.197952,startEncryptionSetupPhaseFor,5.233553417s,AggregationService,5233.553417,5233.596,10467.149417,25.039206,5258.592623
7,2025/07/22,14:41:34.049860,addClient,833ns,AggregationService,0.000833,8085.504,8085.504833,5258.592623,5258.593456
8,2025/07/22,14:41:34.060062,DataSpaceClientService - RegisterClient,12.486834ms,Client2,12.486834,8095.706,8108.192834,5258.593456,5271.080290
9,2025/07/22,14:41:34.068452,HEService - SetParameters,8.336709ms,Client2,8.336709,8104.096,8112.432709,5271.080290,5279.416999


In [86]:
df = px.data.tips()
fig = px.bar(initializiation_data, base="StartMs", x="runtime_ms", y="Service", color='Name', orientation='h',
             hover_data=["EndTime", "StartMs", "runtime_ms"], color_discrete_sequence=px.colors.qualitative.Safe,
             height=400)
fig.show()

In [87]:
request_client_training = data_joined.loc[data_joined["Name"] == "requestClientTraining"].index

print(request_client_training[0])
print(request_client_training[1])

data_after_upload = data_joined[request_client_training[0]:request_client_training[1]]
data_after_upload

17
31


,Date,EndTime,Name,Runtime,Service,runtime_ms,StartMs,EndMs,Start_without_offset,End_without_offset
17,2025/07/22,14:41:53.083988,requestClientTraining,4.722084ms,AggregationService,4.722084,27119.632,27124.354084,10651.106582,10655.828666
18,2025/07/22,14:42:05.271684,HEService - Encrypt,861.965792ms,Client1,861.965792,39307.328,40169.293792,10655.828666,11517.794458
19,2025/07/22,14:42:05.300475,HEService - Encrypt,864.07375ms,Client2,864.073750,39336.119,40200.192750,11517.794458,12381.868208
20,2025/07/22,14:42:06.613421,DataSpaceClientService - UploadData,1.30605325s,Client1,1306.053250,40649.065,41955.118250,12381.868208,13687.921458
21,2025/07/22,14:42:06.638348,DataSpaceClientService - UploadData,1.300053334s,Client2,1300.053334,40673.992,41974.045334,13687.921458,14987.974792
22,2025/07/22,14:42:24.106563,aggregateWeights,49.423458ms,AggregationService,49.423458,58142.207,58191.630458,14987.974792,15037.398250
23,2025/07/22,14:42:24.294186,HEService - PublicKeySwitchGeneration,29.733375ms,Client1,29.733375,58329.830,58359.563375,15037.398250,15067.131625
24,2025/07/22,14:42:24.294211,EncryptionHandler - handlePublicKeySwitch,106.144666ms,Client1,106.144666,58329.855,58435.999666,15067.131625,15173.276291
25,2025/07/22,14:42:24.485745,HEService - PublicKeySwitchGeneration,29.365167ms,Client2,29.365167,58521.389,58550.754167,15173.276291,15202.641458
26,2025/07/22,14:42:24.485772,EncryptionHandler - handlePublicKeySwitch,104.59125ms,Client2,104.591250,58521.416,58626.007250,15202.641458,15307.232708


In [88]:
df = px.data.tips()
fig = px.bar(data_after_upload, base="StartMs", x="runtime_ms", y="Service", color='Name', orientation='h',
             hover_data=["EndTime", "StartMs", "runtime_ms"], color_discrete_sequence=px.colors.qualitative.Safe,
             height=400)
fig.show()